# Windowed MFCC Feature Extraction (IEMOCAP)

This notebook extracts **window-based MFCC features** for real-time style modeling.
Each window becomes one training row (streaming-ready), rather than one row per utterance.

In [1]:
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import librosa
import numpy as np
import pandas as pd


In [2]:
# Configuration
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not locate repository root (missing pyproject.toml).")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from feature_extraction.common import machine_name_from_env, resolve_thread_workers

MACHINE_NAME = machine_name_from_env()
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"  # labels + paths
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"  # wav root folder
OUT_DIR = REPO_ROOT / "extracted_features" / "mfcc"  # output folder
OUT_FILE = "mfcc_features.csv"  # one row per wav
OUT_FILE_NORMALIZED = "mfcc_features_normalized.csv"  # one row per wav (normalized audio)

# Audio + feature params
TARGET_SR = 16_000  # fixed sample rate for consistent features
N_FFT = 1024  # FFT window size
HOP_LENGTH = 256  # hop length between frames
N_MFCC_LIST = (13, 20, 40)  # multiple MFCC granularities
EXCLUDED_EMOTIONS = {"sur", "fea", "oth", "dis"}

OUT_DIR.mkdir(parents=True, exist_ok=True)  # ensure output directory exists
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH_NORMALIZED = OUT_DIR / OUT_FILE_NORMALIZED
OUT_PATH, OUT_PATH_NORMALIZED


(PosixPath('/Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/mfcc/mfcc_features.csv'),
 PosixPath('/Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/mfcc/mfcc_features_normalized.csv'))

In [3]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def normalize_audio_peak(audio: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    # Peak-normalize per file; keep silent/near-silent audio unchanged
    peak = float(np.max(np.abs(audio)))
    if peak <= eps:
        return audio
    return audio / peak


def summarize_matrix(prefix: str, matrix: np.ndarray) -> dict[str, float]:
    # Global stats across all coeffs and frames in the window
    stats: dict[str, float] = {}
    stats[f"{prefix}_frames"] = float(matrix.shape[1])  # frame count in this window
    stats[f"{prefix}_mean"] = float(matrix.mean())
    stats[f"{prefix}_std"] = float(matrix.std())
    stats[f"{prefix}_min"] = float(matrix.min())
    stats[f"{prefix}_max"] = float(matrix.max())
    stats[f"{prefix}_median"] = float(np.median(matrix))
    return stats


def summarize_per_coeff(prefix: str, matrix: np.ndarray) -> dict[str, float]:
    # Per-coefficient stats preserve the spectral envelope shape
    stats: dict[str, float] = {}
    coeff_means = matrix.mean(axis=1)
    coeff_stds = matrix.std(axis=1)
    coeff_mins = matrix.min(axis=1)
    coeff_maxs = matrix.max(axis=1)
    coeff_medians = np.median(matrix, axis=1)
    for idx in range(matrix.shape[0]):
        stats[f"{prefix}_c{idx:02d}_mean"] = float(coeff_means[idx])
        stats[f"{prefix}_c{idx:02d}_std"] = float(coeff_stds[idx])
        stats[f"{prefix}_c{idx:02d}_min"] = float(coeff_mins[idx])
        stats[f"{prefix}_c{idx:02d}_max"] = float(coeff_maxs[idx])
        stats[f"{prefix}_c{idx:02d}_median"] = float(coeff_medians[idx])
    return stats


def compute_mfcc_family(audio: np.ndarray, sr: int) -> dict[int, dict[str, np.ndarray]]:
    # Compute MFCCs + deltas for each MFCC size
    results: dict[int, dict[str, np.ndarray]] = {}
    for n_mfcc in N_MFCC_LIST:
        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=n_mfcc,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            fmin=50,
            fmax=sr // 2,
        )
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        results[n_mfcc] = {"mfcc": mfcc, "d1": delta, "d2": delta2}
    return results


def extract_full_features(
    feature_dict: dict[int, dict[str, np.ndarray]],
) -> dict[str, float]:
    # Summarize MFCC + delta + delta2 over the full utterance
    features: dict[str, float] = {}
    for n_mfcc, matrices in feature_dict.items():
        for key, mat in matrices.items():
            prefix = f"mfcc{n_mfcc}_{key}"
            features.update(summarize_matrix(prefix, mat))
            features.update(summarize_per_coeff(prefix, mat))
    return features


def build_mfcc_dataframe(
    df_meta: pd.DataFrame,
    audio_root: Path,
    normalize: bool = False,
) -> tuple[pd.DataFrame, list[str]]:
    # Build one row per wav with MFCC-family summary features
    cpu_count = os.cpu_count() or 1
    compute_device = "cpu"  # Force CPU for this librosa-based extractor
    num_workers = resolve_thread_workers(MACHINE_NAME)
    progress_min_interval = 1.0

    print(
        f"Compute device: {compute_device} | extractor_backend=cpu | "
        f"machine={MACHINE_NAME} | workers={num_workers}"
    )

    def process_row(row: dict[str, object]) -> tuple[dict[str, float | str | int] | None, str | None]:
        rel_path = str(row["path"])
        audio_path = audio_root / rel_path
        if not audio_path.exists():
            return None, str(audio_path)

        audio, sr = load_audio(audio_path)
        if normalize:
            audio = normalize_audio_peak(audio)

        duration_s = audio.shape[0] / sr  # file duration in seconds
        feature_dict = compute_mfcc_family(audio, sr)  # frame-level MFCCs + deltas
        features = extract_full_features(feature_dict)

        record: dict[str, float | str | int] = {
            "path": rel_path,
            "session": int(row["session"]),
            "method": str(row["method"]),
            "gender": str(row["gender"]),
            "emotion": str(row["emotion"]),
            "n_annotators": int(row["n_annotators"]),
            "agreement": int(row["agreement"]),
            "duration_s": float(duration_s),
        }
        record.update(features)
        return record, None

    rows: list[dict[str, float | str | int]] = []
    missing: list[str] = []
    records = df_meta.to_dict(orient="records")

    if num_workers > 1:
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            mapped = executor.map(process_row, records)
            for record, missing_path in tqdm(mapped, total=len(records), desc="Extracting", unit="file", mininterval=progress_min_interval):
                if missing_path is not None:
                    missing.append(missing_path)
                    continue
                if record is not None:
                    rows.append(record)
    else:
        for record in tqdm(records, total=len(records), desc="Extracting", unit="file", mininterval=progress_min_interval):
            row_result, missing_path = process_row(record)
            if missing_path is not None:
                missing.append(missing_path)
                continue
            if row_result is not None:
                rows.append(row_result)

    print(f"Workers used: {num_workers} (cpu_count={cpu_count}, normalize={normalize})")
    return pd.DataFrame(rows), missing


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()  # normalize labels

# Filter: keep xxx, exclude selected classes, and enforce agreement for labeled classes.
# This keeps unlabeled (xxx) examples while dropping sur/fea/oth/dis.
df = df[~df["emotion"].isin(EXCLUDED_EMOTIONS)].copy()
df = df[(df["emotion"] == "xxx") | (df["agreement"] > 0)].copy()

print(f"Rows after emotion/agreement filters: {len(df):,}")
print("Emotion distribution:")
print(df["emotion"].value_counts())
df.shape


Rows after emotion/agreement filters: 9,887
Emotion distribution:
emotion
xxx    2507
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
Name: count, dtype: int64


(9887, 7)

## Run Options

- Run the **raw-audio** cell to generate `mfcc_features.csv`.
- Run the **normalized-audio** cell to generate `mfcc_features_normalized.csv`.
- These two export cells are independent and can be run separately.


In [5]:
# Raw-audio MFCC export (existing behavior)
window_df_raw, missing_raw = build_mfcc_dataframe(df, AUDIO_ROOT, normalize=False)
window_df_raw.to_csv(OUT_PATH, index=False)

print(f"Saved raw features: {OUT_PATH}")
if missing_raw:
    print(f"Missing audio files: {len(missing_raw)}")
window_df_raw.shape


Compute device: cpu | extractor_backend=cpu | machine=macbook | workers=6


Extracting:   0%|          | 0/9887 [00:00<?, ?file/s]

Workers used: 6 (cpu_count=8, normalize=False)
Saved raw features: /Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/mfcc/mfcc_features.csv


(9887, 1157)

In [6]:
# Peak-normalized-audio MFCC export
window_df_normalized, missing_normalized = build_mfcc_dataframe(df, AUDIO_ROOT, normalize=True)
window_df_normalized.to_csv(OUT_PATH_NORMALIZED, index=False)

print(f"Saved normalized features: {OUT_PATH_NORMALIZED}")
if missing_normalized:
    print(f"Missing audio files: {len(missing_normalized)}")
window_df_normalized.shape


Compute device: cpu | extractor_backend=cpu | machine=macbook | workers=6


Extracting:   0%|          | 0/9887 [00:00<?, ?file/s]

Workers used: 6 (cpu_count=8, normalize=True)
Saved normalized features: /Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/mfcc/mfcc_features_normalized.csv


(9887, 1157)